# Weather Data Fetch — Fixed
Changes from original:
- **Retry with exponential backoff** on HTTP 429 (up to 5 attempts, starting at 60 s wait)
- **Checkpoint file** (`weather_checkpoint.csv`) — saves after every successful node so the run can be interrupted and resumed
- **Nearest-neighbour fallback** — if a node still fails after all retries, its weather is copied from the geographically closest successful node (safe here because all UCSD campus nodes are < 1 km apart)

In [7]:
import pandas as pd
import requests
import time
import math
import os
from datetime import datetime

In [8]:
# Load dataset
df = pd.read_csv('microgrid_final_dataset.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])

min_date = df['DateTime'].min().strftime('%Y-%m-%d')
max_date = df['DateTime'].max().strftime('%Y-%m-%d')
print(f"Data range to fetch: {min_date} to {max_date}")

nodes = df[['NodeID', 'NodeLatitude', 'NodeLongitude']].drop_duplicates().reset_index(drop=True)
display(nodes)

Data range to fetch: 2015-01-01 to 2020-02-29


,NodeID,NodeLatitude,NodeLongitude
0,BioEngineering,32.8801,-117.2340
1,CUP,32.8798,-117.2355
2,CenterHall,32.8784,-117.2373
3,EBU2_A,32.8820,-117.2342
4,EBU2_B,32.8818,-117.2338
5,EastCampus,32.8731,-117.2280
6,GalbraithHall,32.8776,-117.2378
7,GarageFleets,32.8752,-117.2310
8,GeiselLibrary,32.8810,-117.2376
9,GilmanParking,32.8769,-117.2415


In [9]:
CHECKPOINT_FILE = 'weather_checkpoint.csv'
MAX_RETRIES = 5          # attempts per node
INITIAL_BACKOFF = 60     # seconds to wait after first 429
POLITE_DELAY = 3         # seconds between successful requests

# ── Resume support: load already-fetched nodes ──────────────────────────────
if os.path.exists(CHECKPOINT_FILE):
    checkpoint_df = pd.read_csv(CHECKPOINT_FILE, parse_dates=['DateTime'])
    already_done = set(checkpoint_df['NodeID'].unique())
    weather_records = [checkpoint_df]
    print(f"Resuming — {len(already_done)} nodes already in checkpoint: {already_done}")
else:
    already_done = set()
    weather_records = []

failed_nodes = []

# ── Fetch function with retry / backoff ─────────────────────────────────────
def fetch_node(node, lat, lon, min_date, max_date):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={min_date}&end_date={max_date}"
        f"&hourly=temperature_2m,relative_humidity_2m,cloud_cover,shortwave_radiation"
    )
    backoff = INITIAL_BACKOFF
    for attempt in range(1, MAX_RETRIES + 1):
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            if attempt < MAX_RETRIES:
                print(f"  429 on {node} (attempt {attempt}/{MAX_RETRIES}) — waiting {backoff}s...")
                time.sleep(backoff)
                backoff *= 2   # exponential: 60 → 120 → 240 → 480
            else:
                print(f"  ✗ {node} still failing after {MAX_RETRIES} attempts.")
                return None
        else:
            print(f"  ✗ {node}: unexpected status {response.status_code}")
            return None

# ── Main fetch loop ──────────────────────────────────────────────────────────
for _, row in nodes.iterrows():
    node = row['NodeID']
    lat  = row['NodeLatitude']
    lon  = row['NodeLongitude']

    if node in already_done:
        print(f"Skipping {node} (already in checkpoint)")
        continue

    print(f"Fetching {node} ({lat}, {lon})...")
    data = fetch_node(node, lat, lon, min_date, max_date)

    if data is not None:
        hourly = data['hourly']
        temp_df = pd.DataFrame({
            'DateTime':                 pd.to_datetime(hourly['time']),
            'NodeID':                   node,
            'AmbientTemp_C_fetched':    hourly['temperature_2m'],
            'Humidity_pct_fetched':     hourly['relative_humidity_2m'],
            'CloudCoverIndex_fetched':  hourly['cloud_cover'],
            'GHI_Wm2_fetched':          hourly['shortwave_radiation'],
        })
        weather_records.append(temp_df)

        # Save checkpoint after each successful node
        pd.concat(weather_records, ignore_index=True).to_csv(CHECKPOINT_FILE, index=False)
        print(f"  ✓ {node} fetched and checkpointed.")
        time.sleep(POLITE_DELAY)
    else:
        failed_nodes.append(row.to_dict())

print(f"\nFetch complete. Failed nodes: {[n['NodeID'] for n in failed_nodes]}")

Fetching BioEngineering (32.8801, -117.234)...
  ✓ BioEngineering fetched and checkpointed.
Fetching CUP (32.8798, -117.2355)...
  ✓ CUP fetched and checkpointed.
Fetching CenterHall (32.8784, -117.2373)...
  ✓ CenterHall fetched and checkpointed.
Fetching EBU2_A (32.882, -117.2342)...
  ✓ EBU2_A fetched and checkpointed.
Fetching EBU2_B (32.8818, -117.2338)...
  ✓ EBU2_B fetched and checkpointed.
Fetching EastCampus (32.8731, -117.228)...
  ✓ EastCampus fetched and checkpointed.
Fetching GalbraithHall (32.8776, -117.2378)...
  ✓ GalbraithHall fetched and checkpointed.
Fetching GarageFleets (32.8752, -117.231)...
  ✓ GarageFleets fetched and checkpointed.
Fetching GeiselLibrary (32.881, -117.2376)...
  ✓ GeiselLibrary fetched and checkpointed.
Fetching GilmanParking (32.8769, -117.2415)...
  ✓ GilmanParking fetched and checkpointed.
Fetching HopkinsParking (32.8791, -117.2428)...
  ✓ HopkinsParking fetched and checkpointed.
Fetching LeichtagPV (32.8807, -117.2361)...
  ✓ LeichtagPV fet

In [10]:
# ── Nearest-neighbour fallback for any permanently failed nodes ──────────────
# All campus nodes are within ~1 km, so weather is effectively identical.

weather_df = pd.concat(weather_records, ignore_index=True)
successful_nodes = nodes[~nodes['NodeID'].isin([n['NodeID'] for n in failed_nodes])]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # metres
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

for failed in failed_nodes:
    fn_id  = failed['NodeID']
    fn_lat = failed['NodeLatitude']
    fn_lon = failed['NodeLongitude']

    # Find closest successful node
    distances = successful_nodes.apply(
        lambda r: haversine(fn_lat, fn_lon, r['NodeLatitude'], r['NodeLongitude']), axis=1
    )
    nearest_id = successful_nodes.iloc[distances.argmin()]['NodeID']
    dist_m     = distances.min()

    print(f"Filling {fn_id} from nearest node '{nearest_id}' ({dist_m:.0f} m away)")

    donor = weather_df[weather_df['NodeID'] == nearest_id].copy()
    donor['NodeID'] = fn_id
    weather_df = pd.concat([weather_df, donor], ignore_index=True)

print("\nAll nodes now have weather data.")
print(f"Unique NodeIDs in weather_df: {sorted(weather_df['NodeID'].unique())}")


All nodes now have weather data.
Unique NodeIDs in weather_df: ['BioEngineering', 'CUP', 'CenterHall', 'EBU2_A', 'EBU2_B', 'EastCampus', 'GalbraithHall', 'GarageFleets', 'GeiselLibrary', 'GilmanParking', 'HopkinsParking', 'LeichtagPV', 'Mandeville', 'MayerHall', 'MusicBuilding', 'OttersonHall', 'PepperCanyon', 'PriceCenterA', 'PriceCenterB', 'RadyHall', 'RobinsonHall', 'SDSC', 'SocialScience', 'StudentServices']


In [11]:
# ── Merge back onto the 15-minute base dataframe ────────────────────────────
df_sorted      = df.sort_values('DateTime')
weather_sorted = weather_df.sort_values('DateTime')

# Normalise cloud cover: 100 % overcast → 0 (original convention)
weather_sorted['CloudCoverIndex_fetched'] = 1 - (weather_sorted['CloudCoverIndex_fetched'] / 100.0)

mapped_df = pd.merge_asof(
    df_sorted,
    weather_sorted,
    by='NodeID',
    on='DateTime',
    direction='backward'
)

mapped_df['AmbientTemp_C']   = mapped_df['AmbientTemp_C_fetched']
mapped_df['Humidity_pct']    = mapped_df['Humidity_pct_fetched']
mapped_df['CloudCoverIndex'] = mapped_df['CloudCoverIndex_fetched']
mapped_df['GHI_Wm2']         = mapped_df['GHI_Wm2_fetched']

mapped_df.drop(
    columns=['AmbientTemp_C_fetched', 'Humidity_pct_fetched',
             'CloudCoverIndex_fetched', 'GHI_Wm2_fetched'],
    inplace=True
)

# Quick sanity check — no NaNs in weather columns?
weather_cols = ['AmbientTemp_C', 'Humidity_pct', 'CloudCoverIndex', 'GHI_Wm2']
null_counts = mapped_df[weather_cols].isnull().sum()
print("Null counts in weather columns after merge:")
print(null_counts)

display(mapped_df[['DateTime', 'NodeID', 'AmbientTemp_C', 'GHI_Wm2', 'CloudCoverIndex']].head(10))

Null counts in weather columns after merge:
AmbientTemp_C      0
Humidity_pct       0
CloudCoverIndex    0
GHI_Wm2            0
dtype: int64


,DateTime,NodeID,AmbientTemp_C,GHI_Wm2,CloudCoverIndex
0,2015-01-01,LeichtagPV,10.4,211.0,0.84
1,2015-01-01,Mandeville,10.4,211.0,0.84
2,2015-01-01,RadyHall,10.5,211.0,0.84
3,2015-01-01,PepperCanyon,10.5,211.0,0.84
4,2015-01-01,OttersonHall,10.4,211.0,0.84
5,2015-01-01,SDSC,10.5,211.0,0.84
6,2015-01-01,MayerHall,10.5,211.0,0.84
7,2015-01-01,GalbraithHall,10.4,211.0,0.84
8,2015-01-01,RobinsonHall,10.3,211.0,0.84
9,2015-01-01,CUP,10.4,211.0,0.84


In [12]:
# ── Save back to the same CSV ────────────────────────────────────────────────
mapped_df.to_csv('microgrid_final_dataset.csv', index=False)
print("Successfully saved filled dataset to microgrid_final_dataset.csv")

# Optional: clean up the checkpoint file now that we're done
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print(f"Removed checkpoint file: {CHECKPOINT_FILE}")

Successfully saved filled dataset to microgrid_final_dataset.csv
Removed checkpoint file: weather_checkpoint.csv
